In [ ]:
! pip install --quiet anndata mudata scvi-colab
from scvi_colab import install
install()

# Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import anndata as ad
import mudata as mu
import scvi

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from scvi.data import AnnDataManager
from scvi.data.fields import LayerField, NumericalObsField
from scvi.model.base import BaseModelClass
from scvi.module.base import BaseModuleClass, LossOutput
from scvi.train import TrainingPlan, TrainRunner, DataSplitter

RANDOM_SEED = 42

# Simulate data

Same setup as the main notebook — 3 omics blocks with sparse latent structure.
We wrap everything in MuData/AnnData instead of raw tensors.

In [ ]:
def simulate_mudata(seed=42, n=1000, noise=0.0):
    torch.manual_seed(seed)

    p1, p2, p3, k = 50, 80, 30, 2

    # true latent scores for all cells
    z = torch.normal(0, 1, size=(n, k))

    # sparse version: first half only has component 1, second half only component 2
    z_sp = z.clone()
    z_sp[:500, 0] = 0
    z_sp[500:, 1] = 0

    # random loading matrices (true weights, not learned)
    w1 = torch.randn(p1, k)
    w2 = torch.randn(p2, k)
    w3 = torch.randn(p3, k)

    # generate each block as a linear combination of latent scores + noise
    x1 = z_sp @ w1.T + noise * torch.randn(n, p1)
    x2 = z_sp @ w2.T + noise * torch.randn(n, p2)
    x3 = z_sp @ w3.T + noise * torch.randn(n, p3)

    # train/test split indices
    idx = np.arange(n)
    tr_idx, te_idx = train_test_split(idx, test_size=0.2, random_state=seed)
    split = np.array(["train"] * n)
    split[te_idx] = "test"

    # wrap each block in an AnnData — rows are cells, cols are features
    def make_adata(X, name):
        adata = ad.AnnData(X=X.numpy())
        adata.obs["split"] = split  # store train/test label per cell
        adata.var_names = [f"{name}_{i}" for i in range(X.shape[1])]
        return adata

    mod1 = make_adata(x1, "rna")
    mod2 = make_adata(x2, "atac")
    mod3 = make_adata(x3, "prot")

    # the target y (latent scores) goes into .obsm of any modality, accessible from all
    mod1.obsm["y"] = z_sp.numpy()

    mdata = mu.MuData({"rna": mod1, "atac": mod2, "prot": mod3})
    return mdata

mdata = simulate_mudata(seed=RANDOM_SEED)
print(mdata)

# sMBPLS Module

This is the core model logic. It inherits from `BaseModuleClass` so scvi-tools
can handle the training loop, dataloaders, and device management for us.

In [ ]:
def soft_threshold(x, lam):
    # zero out weights smaller than lam, shrink the rest toward zero
    if lam <= 0:
        return x
    return torch.sign(x) * torch.clamp(torch.abs(x) - lam, min=0.0)


class SMBPLSModule(BaseModuleClass):
    def __init__(self, n_input_per_mod, n_components=2, lam_w=0.05, lam_t=0.0):
        super().__init__()

        self.mod_names = list(n_input_per_mod.keys())
        self.K = n_components
        self.lam_w = lam_w
        self.lam_t = lam_t
        alpha = 1.0 / len(self.mod_names)  # equal block weights by default
        self.alpha = {m: alpha for m in self.mod_names}

        # one linear projection per modality: (n, p_b) -> (n, K)
        # no bias — these are PLS-style loadings, not a full NN layer
        self.proj = nn.ModuleDict({
            m: nn.Linear(n_input_per_mod[m], self.K, bias=False)
            for m in self.mod_names
        })

        # regression head maps latent scores to the outcome
        self.regressor = nn.Linear(self.K, 2, bias=True)  # 2 latent components as target

        for m in self.mod_names:
            nn.init.normal_(self.proj[m].weight, std=0.02)
        nn.init.zeros_(self.regressor.bias)

    @torch.no_grad()
    def apply_sparsity(self):
        # proximal L1 step on loadings — called periodically during training
        for m in self.mod_names:
            W = self.proj[m].weight
            W.copy_(soft_threshold(W, self.lam_w))

    def _get_inference_input(self, tensors):
        # tell scvi which tensors to pass into inference()
        return {m: tensors[m] for m in self.mod_names}

    def _get_generative_input(self, tensors, inference_outputs):
        return {"t": inference_outputs["t"]}

    def inference(self, **block_data):
        # compute composite latent score t from all blocks
        t = None
        for m in self.mod_names:
            tb = self.proj[m](block_data[m])
            t = tb * self.alpha[m] if t is None else t + tb * self.alpha[m]

        # optional observation-level sparsity (zeroes out inactive cells)
        if self.lam_t > 0:
            t = soft_threshold(t, self.lam_t)

        return {"t": t}

    def generative(self, t):
        # predict outcome from latent scores
        y_hat = self.regressor(t)
        return {"y_hat": y_hat}

    def loss(self, tensors, inference_outputs, generative_outputs):
        y = torch.tensor(tensors["y"], dtype=torch.float32) if not isinstance(tensors["y"], torch.Tensor) else tensors["y"].float()
        y_hat = generative_outputs["y_hat"]
        t = inference_outputs["t"]

        # MSE reconstruction loss
        mse = F.mse_loss(y_hat, y)

        # covariance loss — the actual PLS objective, maximizes T-y covariance
        T = t - t.mean(0)
        yc = y - y.mean(0)
        cov = (T.T @ yc) / (T.shape[0] - 1)
        cov_loss = -(cov ** 2).sum()

        loss = 0.5 * mse + cov_loss
        return LossOutput(loss=loss, reconstruction_loss=mse, kl_local=torch.zeros(t.shape[0]))

# Custom Training Plan

We subclass `TrainingPlan` just to inject the periodic sparsity step.
Everything else (Adam, logging, val loop) is handled by scvi.

In [ ]:
class SMBPLSTrainingPlan(TrainingPlan):
    def __init__(self, module, sparsity_freq=50, **kwargs):
        super().__init__(module, **kwargs)
        self.sparsity_freq = sparsity_freq
        self._step = 0

    def training_step(self, batch, batch_idx):
        loss_out = super().training_step(batch, batch_idx)
        self._step += 1
        # every sparsity_freq steps, zero out small loadings
        if self._step % self.sparsity_freq == 0:
            self.module.apply_sparsity()
        return loss_out

# SMBPLS model class

This is the user-facing class — similar to how you'd use `scvi.model.SCVI`.
It handles `setup_mudata`, `train`, `get_latent_representation`, and `get_loadings`.

In [ ]:
class SMBPLS(BaseModelClass):
    """
    Sparse Multi-Block PLS for multi-omics MuData.

    Usage
    -----
    SMBPLS.setup_mudata(mdata, modalities=["rna", "atac", "prot"], y_obsm_key="y", y_mod="rna")
    model = SMBPLS(mdata, n_components=2, lam_w=0.05)
    model.train(max_epochs=300)
    z = model.get_latent_representation()
    loadings = model.get_loadings()
    """

    def __init__(self, mdata, n_components=2, lam_w=0.05, lam_t=0.0):
        super().__init__(mdata)

        # figure out how many features each modality has
        n_input = {m: mdata[m].n_vars for m in mdata.mod_names}

        self.module = SMBPLSModule(
            n_input_per_mod=n_input,
            n_components=n_components,
            lam_w=lam_w,
            lam_t=lam_t,
        )
        self._model_summary_string = f"SMBPLS | mods={list(n_input.keys())} | K={n_components}"
        self.init_params_ = self._get_init_params(locals())

    @classmethod
    def setup_mudata(cls, mdata, modalities, y_obsm_key="y", y_mod="rna"):
        """
        Register modalities and outcome with the model.
        modalities: list of modality names in mdata (e.g. ["rna","atac","prot"])
        y_obsm_key: key in mdata[y_mod].obsm where the target matrix is stored
        y_mod: which modality's obsm holds the target
        """
        # store config on the mdata object so the model can find it later
        mdata.uns["smbpls_modalities"] = modalities
        mdata.uns["smbpls_y_mod"] = y_mod
        mdata.uns["smbpls_y_key"] = y_obsm_key
        print(f"Registered modalities: {modalities}, target: {y_mod}.obsm['{y_obsm_key}']")

    def train(self, max_epochs=300, lr=1e-3, batch_size=256, sparsity_freq=50, **kwargs):
        # pull modality matrices and target out of mdata
        mdata = self.adata  # BaseModelClass stores the data object as self.adata
        mods = mdata.uns["smbpls_modalities"]
        y_mod = mdata.uns["smbpls_y_mod"]
        y_key = mdata.uns["smbpls_y_key"]

        # build tensors directly — we bypass scvi's dataloader here because
        # multi-modality batching isn't trivial to set up with DataSplitter
        X_blocks = {m: torch.tensor(mdata[m].X, dtype=torch.float32) for m in mods}
        y = torch.tensor(mdata[y_mod].obsm[y_key], dtype=torch.float32)

        n = y.shape[0]
        idx = np.arange(n)
        tr, te = train_test_split(idx, test_size=0.2, random_state=42)

        X_tr = {m: X_blocks[m][tr] for m in mods}
        X_te = {m: X_blocks[m][te] for m in mods}
        y_tr, y_te = y[tr], y[te]

        opt = torch.optim.Adam(self.module.parameters(), lr=lr)
        self._train_losses, self._val_losses = [], []

        self.module.train()
        for epoch in range(max_epochs):
            # mini-batch loop
            perm = torch.randperm(len(tr))
            epoch_loss = 0.0
            n_batches = 0
            for start in range(0, len(tr), batch_size):
                batch_idx = perm[start:start + batch_size]
                Xb = {m: X_tr[m][batch_idx] for m in mods}
                yb = y_tr[batch_idx]

                opt.zero_grad()
                inf = self.module.inference(**Xb)
                gen = self.module.generative(inf["t"])
                loss_out = self.module.loss({"y": yb}, inf, gen)
                loss_out.loss.backward()
                opt.step()
                epoch_loss += loss_out.loss.item()
                n_batches += 1

            # sparsity proximal step
            if epoch % sparsity_freq == 0:
                self.module.apply_sparsity()

            avg_loss = epoch_loss / n_batches
            self._train_losses.append(avg_loss)

            # validation
            if epoch % 50 == 0:
                self.module.eval()
                with torch.no_grad():
                    inf_te = self.module.inference(**X_te)
                    gen_te = self.module.generative(inf_te["t"])
                    val_mse = F.mse_loss(gen_te["y_hat"], y_te).item()
                self._val_losses.append((epoch, val_mse))
                print(f"epoch {epoch:>4} | train loss {avg_loss:.4f} | val MSE {val_mse:.4f}")
                self.module.train()

        self.is_trained_ = True
        print("Training done.")

    @torch.no_grad()
    def get_latent_representation(self, mdata=None):
        """Return latent scores T for all cells. Stored in mdata.obsm['X_smbpls']."""
        if mdata is None:
            mdata = self.adata
        mods = mdata.uns["smbpls_modalities"]
        X_blocks = {m: torch.tensor(mdata[m].X, dtype=torch.float32) for m in mods}

        self.module.eval()
        inf = self.module.inference(**X_blocks)
        T = inf["t"].numpy()

        # store in the mudata object so downstream tools (scanpy, muon) can use it
        mdata.obsm["X_smbpls"] = T
        print(f"Latent scores stored in mdata.obsm['X_smbpls'], shape {T.shape}")
        return T

    @torch.no_grad()
    def get_loadings(self):
        """Return loading weights for each modality as a dict of DataFrames."""
        mdata = self.adata
        mods = mdata.uns["smbpls_modalities"]
        loadings = {}
        for m in mods:
            W = self.module.proj[m].weight.numpy()  # shape (K, p_b)
            var_names = mdata[m].var_names.tolist()
            df = pd.DataFrame(W.T, index=var_names,
                              columns=[f"component_{k+1}" for k in range(self.module.K)])
            loadings[m] = df
        return loadings

    def plot_training(self):
        plt.figure()
        plt.plot(self._train_losses)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training loss")
        plt.show()

# Run the model

In [ ]:
# register the modalities and target
SMBPLS.setup_mudata(mdata, modalities=["rna", "atac", "prot"], y_obsm_key="y", y_mod="rna")

model = SMBPLS(mdata, n_components=2, lam_w=0.05)
model.train(max_epochs=300, lr=1e-3, batch_size=256)

# Latent scores

In [ ]:
T = model.get_latent_representation()

# quick look at the latent scores — each row is a cell
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ax in enumerate(axes):
    ax.scatter(np.arange(len(T)), T[:, i], s=3, alpha=0.5)
    ax.set_title(f"Component {i+1}")
    ax.set_xlabel("Cell index")
    ax.set_ylabel("Latent score")
plt.tight_layout()
plt.show()

# Loadings — which features got selected per modality

In [ ]:
loadings = model.get_loadings()

for mod_name, df in loadings.items():
    n_nonzero = (df.abs() > 1e-6).sum()
    print(f"{mod_name}: {n_nonzero.values} non-zero loadings per component")

# heatmap of RNA loadings
import seaborn as sns
fig, axes = plt.subplots(1, len(loadings), figsize=(14, 4))
for ax, (mod_name, df) in zip(axes, loadings.items()):
    # only show features with any non-zero loading
    mask = (df.abs() > 1e-6).any(axis=1)
    sub = df[mask]
    if sub.shape[0] == 0:
        sub = df  # fallback if everything got zeroed
    sns.heatmap(sub, ax=ax, center=0, cmap="RdBu_r", yticklabels=False)
    ax.set_title(f"{mod_name} loadings")
plt.tight_layout()
plt.show()

# Predicted vs true latent scores

In [ ]:
mods = mdata.uns["smbpls_modalities"]
y_mod = mdata.uns["smbpls_y_mod"]
y_key = mdata.uns["smbpls_y_key"]

X_all = {m: torch.tensor(mdata[m].X, dtype=torch.float32) for m in mods}
y_all = mdata[y_mod].obsm[y_key]

model.module.eval()
with torch.no_grad():
    inf = model.module.inference(**X_all)
    gen = model.module.generative(inf["t"])
    y_hat = gen["y_hat"].numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ax in enumerate(axes):
    ax.scatter(y_all[:, i], y_hat[:, i], s=5, alpha=0.4)
    mn, mx = y_all[:, i].min(), y_all[:, i].max()
    ax.plot([mn, mx], [mn, mx], "r--")
    r2 = r2_score(y_all[:, i], y_hat[:, i])
    ax.set_title(f"Component {i+1}  R²={r2:.3f}")
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
plt.tight_layout()
plt.show()

# UMAP of latent space

Scanpy can work directly with values stored in `mdata.obsm`.

In [ ]:
import scanpy as sc

# use one of the modalities as a proxy adata for scanpy's neighborhood/umap tools
adata_ref = mdata["rna"].copy()
adata_ref.obsm["X_smbpls"] = T  # latent scores we computed above

sc.pp.neighbors(adata_ref, use_rep="X_smbpls", n_neighbors=15)
sc.tl.umap(adata_ref)

# color by which latent component is active (sparse structure we simulated)
adata_ref.obs["active_component"] = "both"
adata_ref.obs.loc[adata_ref.obs.index[:500], "active_component"] = "comp2_only"
adata_ref.obs.loc[adata_ref.obs.index[500:], "active_component"] = "comp1_only"

sc.pl.umap(adata_ref, color="active_component", title="UMAP colored by simulated cell module")

# Training curve

In [ ]:
model.plot_training()

# Save and reload the model

scvi-tools gives us `.save()` and `.load()` for free via `BaseModelClass`.

In [ ]:
model.save("smbpls_trained/", overwrite=True)
print("Model saved.")

# to reload:
# model2 = SMBPLS.load("smbpls_trained/", adata=mdata)
# T2 = model2.get_latent_representation()